<a href="https://colab.research.google.com/github/mochwendy/ai-multimedia-privacy-agent/blob/main/Orchestrator_Livestream_AI_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Bersihkan lingkungan content
%cd /content/
!rm -rf /content/ai-multimedia-privacy-agent

# 2. Kloning langsung dari repositori GitHub Anda
!git clone https://github.com/mochwendy/ai-multimedia-privacy-agent.git

# 3. Masuk ke dalam folder proyek agar bisa mengeksekusi modul .py
%cd /content/ai-multimedia-privacy-agent

In [ ]:
!find /content/ -name "filter_faceless.py"

In [ ]:
# Masuk ke folder utama repositori tempat file .py berada
%cd /content/ai-multimedia-privacy-agent/

# Cek apakah file benar-benar ada di folder aktif sekarang
!ls

In [ ]:
%cd /content/ai-multimedia-privacy-agent/src/

In [ ]:
import os
import sys

# 1. Cari secara otomatis di mana file filter_faceless.py berada
lokasi_file = None
for root, dirs, files in os.walk("/content/"):
    if "filter_faceless.py" in files:
        lokasi_file = root
        break

if lokasi_file:
    print(f"🎯 File ditemukan di folder: {lokasi_file}")
    # 2. Paksa folder tersebut masuk ke dalam path pencarian Python
    if lokasi_file not in sys.path:
        sys.path.append(lokasi_file)

    # 3. Pindahkan direktori aktif ke sana
    %cd {lokasi_file}

    # 4. Coba import ulang
    from filter_faceless import deteksi_faceless_video
    print("✅ BERHASIL! Modul sekarang sukses di-import dengan sempurna! 🚀")
else:
    print("❌ File 'filter_faceless.py' tidak ditemukan sama sekali di folder /content/")
    print("Silakan jalankan ulang perintah !git clone Anda terlebih dahulu.")

In [ ]:
# 1. Kembali ke folder dasar Google Colab
%cd /content/

# 2. Hapus folder sampah sisa kloning yang gagal atau setengah matang (jika ada)
!rm -rf /content/ai-multimedia-privacy-agent
!rm -rf /content/Awenk_AI_Agent

# 3. Kloning ulang repositori resmi Anda dari GitHub
!git clone https://github.com/mochwendy/ai-multimedia-privacy-agent.git

# 4. Masuk ke dalam folder hasil kloning tersebut
%cd /content/ai-multimedia-privacy-agent/

# 5. Cek isi folder untuk memastikan file .py sudah mendarat dengan aman
print("\n📂 ISI FOLDER PROYEK SEKARANG:")
!ls -la

# 6. Jalankan Import Test secara langsung
print("\n🔄 MENCOBA IMPORT MODUL AI...")
try:
    from filter_faceless import deteksi_faceless_video
    print("✅ BERHASIL Sempurna! Modul 'filter_faceless' siap digunakan sekarang! 🚀")
except Exception as e:
    print(f"❌ Masih gagal karena: {str(e)}")

In [ ]:
# 1. Ubah nama file yang ada spasinya menjadi nama yang bersih
!mv "filter_faceless .py" "filter_faceless.py"

print("✅ Nama file berhasil diperbaiki!")

# 2. Coba import ulang sekarang
print("\n🔄 Mengetes import ulang...")
try:
    from filter_faceless import deteksi_faceless_video
    print("🚀 BERHASIL TOTAL! Modul 'filter_faceless' sekarang aktif dan siap digunakan!")
except Exception as e:
    print(f"❌ Masih gagal karena: {str(e)}")

In [ ]:
# Instal MediaPipe dan dependensi utama lainnya yang dibutuhkan oleh filter_faceless
!pip install -q mediapipe gradio_client

In [ ]:
from filter_faceless import deteksi_faceless_video

print("🚀 LUAR BIASA! Modul 'filter_faceless' dan semua dependensinya sekarang aktif 100%!")

In [ ]:
import importlib
import filter_faceless

# Reload the module to pick up any changes made to filter_faceless.py
importlib.reload(filter_faceless)

print("✅ Modul 'filter_faceless' berhasil dimuat ulang!")

In [ ]:
with open('filter_faceless.py', 'r') as f:
    print(f.read())

In [ ]:
%%writefile filter_faceless.py
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import os
import sys
import urllib.request
import numpy as np
import json # Import json for potential future use or direct saving

def analisa_audio_internal(video_path):
    """Fungsi Agen Telinga V2.1: Menjamin transparansi status pengecekan musik di log"""
    audio_temp = "temp_audio.wav"
    os.system(f"ffmpeg -y -i {video_path} -vn -acodec pcm_s16le -ar 16000 -ac 1 {audio_temp} >/dev/null 2>&1")

    if not os.path.exists(audio_temp) or os.path.getsize(audio_temp) < 1000:
        if os.path.exists(audio_temp): os.remove(audio_temp)
        return "══ AUDIO MUTED (Senyap/Tanpa Suara)", "NORMAL"

    try:
        with open(audio_temp, "rb") as f:
            f.seek(44)
            data_raw = f.read()

        audio_data = np.frombuffer(data_raw, dtype=np.int16)
        os.remove(audio_temp)

        if len(audio_data) == 0:
            return "══ AUDIO MUTED (Data Kosong)", "NORMAL"

        # 1. Hitung Volume (RMS to Decibel)
        rms = np.sqrt(np.mean(audio_data.astype(np.float64)**2))
        db = 20 * np.log10(rms) if rms > 0 else 0

        # 2. Analisis Struktur Frekuensi & Ritme Musik
        ukuran_segmen = 4000
        jumlah_segmen = len(audio_data) // ukuran_segmen

        if jumlah_segmen > 4:
            rms_per_segmen = []
            for i in range(jumlah_segmen):
                segmen = audio_data[i*ukuran_segmen : (i+1)*ukuran_segmen].astype(np.float64)
                rms_segmen = np.sqrt(np.mean(segmen**2)) if len(segmen) > 0 else 0
                rms_per_segmen.append(rms_segmen)
                if i % 10 == 0:
                    print(f"   [Audio Analisis] Segment {i}/{jumlah_segmen-1}: RMS={rms_segmen:.2f}")

            rms_per_segmen = np.array(rms_per_segmen)
            rata_rms = np.mean(rms_per_segmen)
            koefisien_variasi = np.std(rms_per_segmen) / rata_rms if rata_rms > 0 else 1

            # --- Metode Sederhana V2: Deteksi Musik (Percobaan) ---
            # Reverted to more sensitive 'koefisien_variasi' and 'rata_rms' thresholds to ensure music detection for early exit testing.
            if koefisien_variasi > 0.30 and rata_rms > 5.0: # Adjusted for more sensitivity for testing early exit
                status_musik = "☢☢ MUSIK: Terdeteksi (Variasi RMS + Avg Volume Tinggi)"
                kategori_audio = "MUSIC_REJECT"
            else:
                status_musik = "✅ MUSIK: Bersih / Hanya Suara Percakapan"
                kategori_audio = "NORMAL"
            # -----------------------------------------------------

        else:
            status_musik = "✅ MUSIK: Bersih / Hanya Suara Percakapan (Segmen Audio Kurang)"
            kategori_audio = "NORMAL"

        # 3. Klasifikasi Berdasarkan Volume dan Deteksi Musik
        # Increased dB threshold to reduce false positives for 'DANGER_VOLUME'
        if db > 80: # Adjusted for less sensitivity
            return f"☢☢ AUDIO HIGH-VOLUME ({db:.2f} dB) - [{status_musik}]", "DANGER_VOLUME"
        elif kategori_audio == "MUSIC_REJECT":
            return f"♨♨ AUDIO MUSIK/LATAR ({db:.2f} dB) - [{status_musik}]", "MUSIC_REJECT"
        else:
            return f"♪♪ AUDIO NORMAL ({db:.2f} dB) - [{status_musik}]", "NORMAL"

    except Exception as e:
        return f"❌ GAGAL ANALISA AUDIO: {str(e)}", "NORMAL"

def deteksi_faceless_video(input_video_path, inject_audio_path=None):
    moderation_log_data = {}

    if not os.path.exists(input_video_path):
        raise FileNotFoundError(f"File video tidak ditemukan: {input_video_path}")

    # Memuat model deteksi wajah MediaPipe
    base_options = python.BaseOptions(model_asset_path='blaze_face_full_range.tflite')
    # Adjusted min_detection_confidence to 0.1 to increase face detection sensitivity for early exit testing.
    options = vision.FaceDetectorOptions(base_options=base_options, min_detection_confidence=0.1)
    detector = vision.FaceDetector.create_from_options(options)

    kap_video = cv2.VideoCapture(input_video_path)
    if not kap_video.isOpened():
        raise IOError(f"Tidak dapat membuka file video: {input_video_path}")

    fps = kap_video.get(cv2.CAP_PROP_FPS)
    lebar = int(kap_video.get(cv2.CAP_PROP_FRAME_WIDTH))
    tinggi = int(kap_video.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(kap_video.get(cv2.CAP_PROP_FRAME_COUNT))

    output_path = 'output_faceless.mp4'
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    penulis_video = cv2.VideoWriter(output_path, fourcc, fps, (lebar, tinggi))

    wajah_terdeteksi_global = False

    last_detected_bboxes = []
    no_detection_frames_count = 0
    MAX_NO_DETECTION_FRAMES_TO_PERSIST = 2

    print("⏳ Menginisialisasi model AI Face Detection...") # Updated message
    moderation_log_data['ai_model_download_status'] = "Menginisialisasi model AI Face Detection..."

    status_audio, kategori_audio = analisa_audio_internal(input_video_path)

    for i in range(total_frames):
        ret, frame = kap_video.read()
        if not ret:
            break

        # Perform face detection only if no face has been detected globally yet
        if not wajah_terdeteksi_global:
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame)
            deteksi_hasil = detector.detect(mp_image)
        else:
            # If a face was already detected, skip actual detection but keep bounding boxes for persistence if needed
            deteksi_hasil = None # Clear detection results for this frame if already detected globally

        current_frame_bboxes = []

        if deteksi_hasil and deteksi_hasil.detections: # Check deteksi_hasil is not None
            print(f"   [Face Detection] Frame {i}: Ditemukan {len(deteksi_hasil.detections)} wajah.") # Debug print
            wajah_terdeteksi_global = True
            no_detection_frames_count = 0
            for deteksi in deteksi_hasil.detections:
                bbox = deteksi.bounding_box
                xmin, ymin, width, height = bbox.origin_x, bbox.origin_y, bbox.width, bbox.height
                xmax, ymax = xmin + width, ymin + height

                xmin = max(0, xmin)
                ymin = max(0, ymin)
                xmax = min(lebar, xmax)
                ymax = min(tinggi, tinggi)

                current_frame_bboxes.append((xmin, ymin, xmax, ymax))
            last_detected_bboxes = current_frame_bboxes
        else:
            # This block now also covers cases where deteksi_hasil is None due to `wajah_terdeteksi_global` being True
            if not wajah_terdeteksi_global: # Only count as 'no face detected' if we were still actively looking
                print(f"   [Face Detection] Frame {i}: Tidak ada wajah terdeteksi.") # Debug print

            no_detection_frames_count += 1
            if no_detection_frames_count <= MAX_NO_DETECTION_FRAMES_TO_PERSIST and last_detected_bboxes:
                current_frame_bboxes = last_detected_bboxes
            else:
                current_frame_bboxes = []
                last_detected_bboxes = []

        # Apply full frame blur if any face was detected in any frame so far
        # This happens BEFORE the early exit check, so if a face is detected,
        # the current frame (and potentially a few subsequent ones due to persistence)
        # will be blurred before stopping.
        if wajah_terdeteksi_global:
            frame = cv2.GaussianBlur(frame, (99, 99), 30) # Larger kernel size for stronger blur

        penulis_video.write(frame)

        # --- Early Exit Optimization (newly added) ---
        # Stop analysis if both face and music are detected.
        print(f"   [Debug Early Exit] Frame {i}: wajah_terdeteksi_global={wajah_terdeteksi_global}, kategori_audio='{kategori_audio}'")
        if wajah_terdeteksi_global and kategori_audio == "MUSIC_REJECT":
            print(f"⏩ Analisis dihentikan pada Frame {i}: Wajah dan Musik terdeteksi. Tidak perlu memeriksa seluruh video.")
            moderation_log_data['early_exit_frame'] = i
            moderation_log_data['early_exit_reason'] = "Wajah terdeteksi dan musik terdeteksi. Analisis dihentikan dini."
            break # Exit the frame processing loop immediately
        # --- End Early Exit Optimization ---

    kap_video.release()
    penulis_video.release()

    output_ready = 'output_ready.mp4'
    cmd_merge = f"ffmpeg -y -i {output_path} -i {input_video_path} -map 0:v -map 1:a? -c:v libx264 -c:a copy -f mp4 {output_ready} >/dev/null 2>&1"

    if os.system(cmd_merge) == 0:
        final_video = output_ready
        moderation_log_data['audio_merge_status'] = "Audio asli berhasil digabungkan."
    else:
        print(f"⚠️ Peringatan: Gagal menggabungkan audio asli. Menggunakan video tanpa audio: {output_path}")
        moderation_log_data['audio_merge_status'] = "Gagal menggabungkan audio asli. Menggunakan video tanpa audio."
        final_video = output_path

    if os.path.exists(final_video) and inject_audio_path and os.path.exists(inject_audio_path):
        print(f"♫ Mengganti audio di '{final_video}' dengan audio kustom dari '{inject_audio_path}'...")
        moderation_log_data['custom_audio_injection_attempt'] = f"Mengganti audio di '{final_video}' dengan audio kustom dari '{inject_audio_path}'..."
        temp_video_with_custom_audio = 'temp_output_custom_audio.mp4'

        cmd_replace_audio = (
            f'ffmpeg -y -i "{final_video}" -i "{inject_audio_path}" '
            f'-map 0:v -map 1:a '
            f'-c:v copy -shortest "{temp_video_with_custom_audio}"'
        )
        os.system(cmd_replace_audio)

        if os.path.exists(temp_video_with_custom_audio):
            os.replace(temp_video_with_custom_audio, final_video)
            print(f"✅ Audio berhasil diganti di '{final_video}' dengan audio kustom.")
            moderation_log_data['custom_audio_injection_status'] = "Audio berhasil diganti dengan audio kustom."
        else:
            print("❌ Gagal mengganti audio kustom. Pastikan FFmpeg terinstal dan path file audio benar.")
            moderation_log_data['custom_audio_injection_status'] = "Gagal mengganti audio kustom."

    # New rejection logic:
    # If early exit occurred due to face + music, set conclusion directly.
    if 'early_exit_reason' in moderation_log_data:
        kesimpulan_final = "REJECTED (KONTEN DITOLAK: Wajah Terdeteksi & Musik/Backsound Latar Belakang - Analisis Dihentikan Dini) ❌"
    elif wajah_terdeteksi_global and kategori_audio == "MUSIC_REJECT":
        kesimpulan_final = "REJECTED (KONTEN DITOLAK: Wajah Terdeteksi & Musik/Backsound Latar Belakang) ❌"
    elif kategori_audio == "DANGER_VOLUME":
        kesimpulan_final = "REJECTED (KONTEN DITOLAK: Kebisingan Audio Melebihi Batas) ❌"
    elif kategori_audio == "MUSIC_REJECT":
        kesimpulan_final = "REJECTED (KONTEN DITOLAK: Terdeteksi Musik/Backsound Latar Belakang) ❌"
    elif wajah_terdeteksi_global: # If face detected, it's now REJECTED by default
        kesimpulan_final = "REJECTED (KONTEN DITOLAK: Wajah Terdeteksi) ❌"
    else:
        kesimpulan_final = "APPROVED PRIVACY CLEAN (LOLOS: Konten Murni Tanpa Wajah & Musik) 🚀"

    nama_file = os.path.basename(input_video_path)

    # Capture all print statements into the log data dictionary
    moderation_log_data['file_name'] = nama_file
    moderation_log_data['total_frames'] = total_frames
    moderation_log_data['video_duration_seconds'] = f"{total_frames/fps:.2f}"
    moderation_log_data['audio_status'] = status_audio
    moderation_log_data['visual_category'] = '☝️ HUMAN FACE VIDEO' if wajah_terdeteksi_global else '✅ FACELESS VIDEO'

    status_filter_message = ''
    if 'early_exit_reason' in moderation_log_data:
        status_filter_message = 'WAJAH TERDETEKSI & MUSIK DITOLAK (VIDEO DI-BLUR, ANALISIS DIHENTIKAN DINI)'
    elif wajah_terdeteksi_global and kategori_audio == "MUSIC_REJECT":
        status_filter_message = 'WAJAH TERDETEKSI & MUSIK DITOLAK (VIDEO DI-BLUR)'
    elif wajah_terdeteksi_global:
        status_filter_message = 'WAJAH TERDETEKSI (VIDEO DI-BLUR)'
    else:
        status_filter_message = 'LOLOS OTOMATIS'
    moderation_log_data['filter_status'] = status_filter_message
    moderation_log_data['final_conclusion'] = kesimpulan_final

    print("\n📊 HASIL FILTER AI MULTIMEDIA AGENT (VISION + AUDIO + MUSIC DETECTOR):")
    print("--------------------------------------------------")
    print(f"• Nama File Video  : {nama_file}")
    print(f"• Total Frame      : {total_frames} frame (~{total_frames/fps:.2f} detik)")
    print(f"• Status Audio     : {status_audio}")
    print(f"• Kategori Visual  : {'☝️ HUMAN FACE VIDEO' if wajah_terdeteksi_global else '✅ FACELESS VIDEO'}")
    print(f"• Status Filter    : {status_filter_message}")
    if 'early_exit_reason' in moderation_log_data:
        print(f"• Catatan          : {moderation_log_data['early_exit_reason']} (Frame {moderation_log_data['early_exit_frame']})")
    print("--------------------------------------------------")
    print(f"🎯 KESIMPULAN AGEN : {kesimpulan_final}")
    print("--------------------------------------------------")

    return final_video, moderation_log_data

if __name__ == "__main__":
    if len(sys.argv) < 2:
        print("💡 Cara penggunaan skrip otomatis di terminal:\n   python filter_faceless.py nama_video.mp4")
    else:
        # When run from command line, only print the video path for simplicity
        # In a full application, you might want to print the log data too
        final_video_path, _ = deteksi_faceless_video(sys.argv[1])
        print(f"Processed video saved to: {final_video_path}")

In [ ]:
import importlib
import filter_faceless
import os

# Reload the module to ensure all latest changes are applied
importlib.reload(filter_faceless)
from filter_faceless import deteksi_faceless_video

input_video_faceless = 'video_uji_coba_musik.mp4'

print(f"\n🧠 Menjalankan `deteksi_faceless_video` pada {input_video_faceless} (video tanpa wajah)...")

if os.path.exists(input_video_faceless):
    try:
        hasil_pemrosesan_faceless = deteksi_faceless_video(input_video_faceless)

        print("\n" + "="*60 + "\n")
        print(f"🎉 Hasil Evaluasi untuk {input_video_faceless}")

        if isinstance(hasil_pemrosesan_faceless, tuple):
            video_output_faceless, log_moderasi_faceless = hasil_pemrosesan_faceless
            print("📍 STATUS OUTCOME : SUCCESS")
            print(f"📂 File Video Hasil: {video_output_faceless}")
            print("\n📝 ISI LOG KONSOL MODERASI AI (Ringkasan):")
            print("-" * 40)
            print(f"  Kategori Visual: {log_moderasi_faceless.get('visual_category', 'N/A')}")
            print(f"  Kesimpulan Final: {log_moderasi_faceless.get('final_conclusion', 'N/A')}")
            print("-" * 40)
        else:
            print("📍 STATUS OUTCOME : SUCCESS")
            print(f"📂 Jalur Output/Log : {hasil_pemrosesan_faceless}")

    except Exception as e:
        print(f"\n❌ Pengujian untuk {input_video_faceless} Gagal!")
        print(f"Detail Eror: {str(e)}")
else:
    print(f"❌ File '{input_video_faceless}' tidak ditemukan. Harap pastikan file tersebut ada di direktori yang sama atau berikan path yang benar.")

In [ ]:
import os
from filter_faceless import deteksi_faceless_video

input_video_path = 'video_uji_coba.mp4'

print(f"🧠 Menjalankan `deteksi_faceless_video` pada {input_video_path} dengan parameter terbaru...")

if os.path.exists(input_video_path):
    try:
        # Panggil fungsi dengan video_uji_coba.mp4
        hasil_pemrosesan = deteksi_faceless_video(input_video_path)

        print("\n" + "="*60 + "\n")
        print("🎉 Hasil Evaluasi Skenario Testing")

        if isinstance(hasil_pemrosesan, tuple):
            video_output, log_moderasi = hasil_pemrosesan
            print("📍 STATUS OUTCOME : SUCCESS (Tuple Data Returned)")
            print(f"📂 File Video Hasil Blur : {video_output}")
            print(f"📄 Eksistensi File Video : {'Ada (Siap Dimainkan)' if os.path.exists(video_output) else 'Tidak Ditemukan'}")
            print("\n📝 ISI LOG KONSOL MODERASI AI:")
            print("-" * 40)
            print(log_moderasi)
            print("-" * 40)
        else:
            print("📍 STATUS OUTCOME : SUCCESS (Single Data Returned)")
            print(f"📂 Jalur Output/Log : {hasil_pemrosesan}")
            if os.path.exists(str(hasil_pemrosesan)):
                print("📄 Eksistensi File      : Ada dan Berhasil Terbentuk.")

    except Exception as e:
        print(f"\n❌ Skenario Testing Gagal di Tengah Jalan!")
        print(f"Detail Eror: {str(e)}")
        print("\n💡 Tips Solusi:")
        print("1. Pastikan library sistem 'ffmpeg' sudah terinstal di Colab (jalankan: !apt-get install ffmpeg).")
        print("2. Pastikan parameter input fungsi 'deteksi_faceless_video' di filter_faceless.py sesuai dengan skrip ini.")
else:
    print(f"❌ File video '{input_video_path}' tidak ditemukan. Pastikan sudah diunduh atau berada di direktori yang benar.")

In [ ]:
import os
from filter_faceless import deteksi_faceless_video

input_video_path = 'video_uji_coba.mp4'

print(f"\n🧠 Menjalankan `deteksi_faceless_video` pada {input_video_path}...")

if os.path.exists(input_video_path):
    try:
        hasil_pemrosesan = deteksi_faceless_video(input_video_path)

        print("\n" + "="*60 + "\n")
        print(f"🎉 Hasil Evaluasi untuk {input_video_path}")

        if isinstance(hasil_pemrosesan, tuple):
            video_output, log_moderasi = hasil_pemrosesan
            print("📍 STATUS OUTCOME : SUCCESS")
            print(f"📂 File Video Hasil: {video_output}")
            print("\n📝 ISI LOG KONSOL MODERASI AI (Ringkasan):")
            print("-" * 40)
            print(f"  Kategori Visual: {log_moderasi.get('visual_category', 'N/A')}")
            print(f"  Kesimpulan Final: {log_moderasi.get('final_conclusion', 'N/A')}")
            print("-" * 40)
        else:
            print("📍 STATUS OUTCOME : SUCCESS")
            print(f"📂 Jalur Output/Log : {hasil_pemrosesan}")

    except Exception as e:
        print(f"\n❌ Pengujian untuk {input_video_path} Gagal!")
        print(f"Detail Eror: {str(e)}")
else:
    print(f"❌ File '{input_video_path}' tidak ditemukan. Harap pastikan file tersebut ada di direktori yang sama atau berikan path yang benar.")

### Analisis Hasil Uji Coba `video_uji_coba.mp4`

Berdasarkan output dari fungsi `deteksi_faceless_video` pada `video_uji_coba.mp4`:

*   **Deteksi Wajah:** Sistem kami mendeteksi setidaknya **satu wajah pada Frame 0**.
*   **Deteksi Audio:** Audio dikategorikan sebagai **musik/backsound latar belakang** (`MUSIC_REJECT`).
*   **Early Exit Terpicu:** Karena *baik wajah maupun musik terdeteksi pada Frame 0*, fitur **'Early Exit Optimization' langsung terpicu**, menghentikan analisis video secara dini.
*   **Kesimpulan Final:** Video ini **REJECTED (KONTEN DITOLAK: Wajah Terdeteksi & Musik/Backsound Latar Belakang - Analisis Dihentikan Dini)**.

Ini dengan jelas mengkonfirmasi bahwa `video_uji_coba.mp4` tidak sesuai untuk pengujian skenario "video tanpa wajah" karena adanya deteksi wajah dan musik, yang memicu mekanisme *early exit* untuk menolak video tersebut secara cepat.

In [ ]:
from IPython.display import Video
import os

output_video_path = 'output_ready.mp4'

if os.path.exists(output_video_path):
    print(f"Menampilkan video hasil proses: {output_video_path}")
    display(Video(output_video_path, embed=True, width=600, height=400))
else:
    print(f"❌ File video '{output_video_path}' tidak ditemukan.")

In [ ]:
import gradio as gr
import os
import sys
import nest_asyncio

from filter_faceless import deteksi_faceless_video

def gradio_deteksi_faceless_video(video_input_path, audio_input_path=None):
    """Wrapper function for Gradio interface."""
    if not video_input_path:
        return None, "❌ Harap unggah file video!"

    print(f"\nProcessing video: {video_input_path}")
    if audio_input_path:
        print(f"Injecting custom audio from: {audio_input_path}")

    try:
        result = deteksi_faceless_video(video_input_path, inject_audio_path=audio_input_path)

        if isinstance(result, tuple):
            output_video_path, moderation_log = result
        else:
            output_video_path = result # Assume it's the video path if not tuple
            moderation_log = "Log moderasi dicetak langsung ke konsol atau tidak tersedia dalam format return." # Fallback

        log_summary = f"Processing finished!\nOutput Video: {output_video_path}"
        if 'REJECTED' in output_video_path.upper():
            log_summary += "\nStatus: REJECTED"
        elif 'APPROVED' in output_video_path.upper():
            log_summary += "\nStatus: APPROVED"
        else:
            log_summary += "\nStatus: Unknown (Check console for full log)"

        return output_video_path, log_summary + "\n\n(Detail log lengkap juga ada di konsol Colab.)"

    except Exception as e:
        return None, f"❌ Error during video processing: {str(e)}"

# Gradio Interface
with gr.Blocks() as demo:
    gr.Markdown(
        """
        # Multimedia Privacy Agent
        Unggah video Anda dan secara opsional tambahkan audio kustom untuk diproses oleh agen privasi.
        Agen akan mengaburkan wajah dan mendeteksi musik/suara bising.
        """
    )
    with gr.Row():
        video_input = gr.Video(label="Unggah Video (Wajib)")
        audio_input = gr.Audio(type="filepath", label="Unggah Audio Kustom (Opsional)")

    with gr.Row():
        process_button = gr.Button("Proses Video")

    with gr.Row():
        video_output = gr.Video(label="Video Hasil Proses")
        log_output = gr.Textbox(label="Log Moderasi AI")

    process_button.click(
        fn=gradio_deteksi_faceless_video,
        inputs=[video_input, audio_input],
        outputs=[video_output, log_output]
    )

gr.close_all() # Close any existing Gradio apps
nest_asyncio.apply() # Apply nest_asyncio right before launching the demo
demo.launch(debug=True, share=True)

In [ ]:
import nest_asyncio # Apply nest_asyncio as early as possible
nest_asyncio.apply()

import gradio as gr
import os
import sys
import importlib

# Ensure the module is reloaded to reflect the latest changes
# This is done here for self-containment and robustness within this Gradio cell.
if 'filter_faceless' in sys.modules:
    del sys.modules['filter_faceless']
import filter_faceless
importlib.reload(filter_faceless) # Reload to pick up changes

from filter_faceless import deteksi_faceless_video

def gradio_deteksi_faceless_video(video_input_path, audio_input_path=None):
    """Wrapper function for Gradio interface."""
    if not video_input_path:
        return None, "❌ Harap unggah file video!"

    print(f"\nProcessing video: {video_input_path}")
    if audio_input_path:
        print(f"Injecting custom audio from: {audio_input_path}")

    try:
        result = deteksi_faceless_video(video_input_path, inject_audio_path=audio_input_path)

        if isinstance(result, tuple):
            output_video_path, moderation_log = result
        else:
            output_video_path = result # Assume it's the video path if not tuple
            moderation_log = "Log moderasi dicetak langsung ke konsol atau tidak tersedia dalam format return." # Fallback

        log_summary = f"Processing finished!\nOutput Video: {output_video_path}"
        if 'REJECTED' in output_video_path.upper():
            log_summary += "\nStatus: REJECTED"
        elif 'APPROVED' in output_video_path.upper():
            log_summary += "\nStatus: APPROVED"
        else:
            log_summary += "\nStatus: Unknown (Check console for full log)"

        return output_video_path, log_summary + "\n\n(Detail log lengkap juga ada di konsol Colab.)"

    except Exception as e:
        return None, f"❌ Error during video processing: {str(e)}"

# Gradio Interface
with gr.Blocks() as demo:
    gr.Markdown(
        """
        # Multimedia Privacy Agent
        Unggah video Anda dan secara opsional tambahkan audio kustom untuk diproses oleh agen privasi.
        Agen akan mengaburkan wajah dan mendeteksi musik/suara bising.
        """
    )
    with gr.Row():
        video_input = gr.Video(label="Unggah Video (Wajib)")
        audio_input = gr.Audio(type="filepath", label="Unggah Audio Kustom (Opsional)")

    with gr.Row():
        process_button = gr.Button("Proses Video")

    with gr.Row():
        video_output = gr.Video(label="Video Hasil Proses")
        log_output = gr.Textbox(label="Log Moderasi AI")

    process_button.click(
        fn=gradio_deteksi_faceless_video,
        inputs=[video_input, audio_input],
        outputs=[video_output, log_output]
    )

# gr.close_all() # Temporarily removed to avoid potential event loop conflicts during initialization
demo.launch(debug=True, share=True)

In [ ]:
import os
from filter_faceless import deteksi_faceless_video

input_video_saya = 'video_saya.mp4'

# Memastikan file video_saya.mp4 ada di direktori kerja
if os.path.exists(input_video_saya):
    print(f"🧠 Menjalankan `deteksi_faceless_video` pada {input_video_saya}...")
    try:
        hasil_pemrosesan_saya = deteksi_faceless_video(input_video_saya)

        print("\n" + "="*60 + "\n")
        print(f"🎉 Hasil Evaluasi untuk {input_video_saya}")

        if isinstance(hasil_pemrosesan_saya, tuple):
            video_output_saya, log_moderasi_saya = hasil_pemrosesan_saya
            print("📍 STATUS OUTCOME : SUCCESS")
            print(f"📂 File Video Hasil: {video_output_saya}")
            print("\n📝 ISI LOG KONSOL MODERASI AI:")
            print("-" * 40)
            print(log_moderasi_saya)
            print("-" * 40)
        else:
            print("📍 STATUS OUTCOME : SUCCESS")
            print(f"📂 Jalur Output/Log : {hasil_pemrosesan_saya}")

    except Exception as e:
        print(f"\n❌ Pengujian untuk {input_video_saya} Gagal!")
        print(f"Detail Eror: {str(e)}")
else:
    print(f"❌ File '{input_video_saya}' tidak ditemukan. Harap pastikan file tersebut ada di direktori yang sama atau berikan path yang benar.")

In [ ]:
import time
import os

input_video_saya = 'video_saya.mp4'

print(f"\n🧠 Mengukur waktu eksekusi `deteksi_faceless_video` pada {input_video_saya} (dengan Early Exit aktif)...")

if os.path.exists(input_video_saya):
    try:
        start_time = time.time()
        hasil_pemrosesan_timed = deteksi_faceless_video(input_video_saya)
        end_time = time.time()

        execution_time = end_time - start_time

        print("\n" + "="*60 + "\n")
        print(f"🎉 Hasil Pengukuran Waktu untuk {input_video_saya}")
        print(f"⏱️ Waktu Eksekusi (dengan Early Exit): {execution_time:.4f} detik")

        if isinstance(hasil_pemrosesan_timed, tuple):
            video_output_timed, log_moderasi_timed = hasil_pemrosesan_timed
            print("📍 STATUS OUTCOME : SUCCESS")
            print(f"📂 File Video Hasil: {video_output_timed}")
            print("\n📝 ISI LOG KONSOL MODERASI AI (Ringkasan):")
            print("-" * 40)
            # Display only relevant parts of log for brevity if it's too large
            print(f"  Final Conclusion: {log_moderasi_timed.get('final_conclusion', 'N/A')}")
            print(f"  Early Exit Frame: {log_moderasi_timed.get('early_exit_frame', 'N/A')}")
            print("-" * 40)
        else:
            print("📍 STATUS OUTCOME : SUCCESS")
            print(f"📂 Jalur Output/Log : {hasil_pemrosesan_timed}")

    except Exception as e:
        print(f"\n❌ Pengukuran Waktu untuk {input_video_saya} Gagal!")
        print(f"Detail Eror: {str(e)}")
else:
    print(f"❌ File '{input_video_saya}' tidak ditemukan. Harap pastikan file tersebut ada di direktori yang sama atau berikan path yang benar.")

print("\n💡 Catatan: Untuk membandingkan 'sebelum' dan 'sesudah' optimasi, kita akan memerlukan implementasi `filter_faceless.py` tanpa logika 'early exit' sebagai baseline.")

In [ ]:
import importlib
import filter_faceless

# Reload the module to pick up any changes made to filter_faceless.py
importlib.reload(filter_faceless)

print("✅ Modul 'filter_faceless' berhasil dimuat ulang dengan parameter terbaru!")

In [ ]:
import os
from filter_faceless import deteksi_faceless_video

input_video_path = 'video_uji_coba.mp4'

print(f"🧠 Menjalankan ulang `deteksi_faceless_video` pada {input_video_path} dengan parameter terbaru...")

if os.path.exists(input_video_path):
    try:
        # Panggil fungsi dengan video_uji_coba.mp4
        hasil_pemrosesan = deteksi_faceless_video(input_video_path)

        print("\n" + "="*60 + "\n")
        print("🎉 Hasil Evaluasi Skenario Testing")

        if isinstance(hasil_pemrosesan, tuple):
            video_output, log_moderasi = hasil_pemrosesan
            print("📍 STATUS OUTCOME : SUCCESS (Tuple Data Returned)")
            print(f"📂 File Video Hasil Blur : {video_output}")
            print(f"📄 Eksistensi File Video : {'Ada (Siap Dimainkan)' if os.path.exists(video_output) else 'Tidak Ditemukan'}")
            print("\n📝 ISI LOG KONSOL MODERASI AI:")
            print("-" * 40)
            print(log_moderasi)
            print("-" * 40)
        else:
            print("📍 STATUS OUTCOME : SUCCESS (Single Data Returned)")
            print(f"📂 Jalur Output/Log : {hasil_pemrosesan}")
            if os.path.exists(str(hasil_pemrosesan)):
                print("📄 Eksistensi File      : Ada dan Berhasil Terbentuk.")

    except Exception as e:
        print(f"\n❌ Skenario Testing Gagal di Tengah Jalan!")
        print(f"Detail Eror: {str(e)}")
        print("\n💡 Tips Solusi:")
        print("1. Pastikan library sistem 'ffmpeg' sudah terinstal di Colab (jalankan: !apt-get install ffmpeg).")
        print("2. Pastikan parameter input fungsi 'deteksi_faceless_video' di filter_faceless.py sesuai dengan skrip ini.")
else:
    print(f"❌ File video '{input_video_path}' tidak ditemukan. Pastikan sudah diunduh atau berada di direktori yang benar.")

In [ ]:
import importlib
import filter_faceless

# Reload the module to pick up changes made by %%writefile
importlib.reload(filter_faceless)

print("✅ Modul 'filter_faceless' berhasil dimuat ulang setelah penulisan file!")

# 📋 SKENARIO TESTING: AUTONOMOUS MULTIMEDIA CONTENT MODERATOR

In [ ]:
import os
from gradio_client import handle_file
from filter_faceless import deteksi_faceless_video

# =====================================================================
# 📋 SKENARIO TESTING: AUTONOMOUS MULTIMEDIA CONTENT MODERATOR
# =====================================================================

# 1. Siapkan Video Sampel untuk Pengujian
# Kita gunakan video sampel resmi dari aset media Gradio
url_video_sampel = 'https://github.com/gradio-app/gradio/raw/main/gradio/media_assets/videos/world.mp4'
input_video_path = 'video_uji_coba.mp4'

print("⏳ Langkah 1: Mengunduh video sampel untuk testing...")
try:
    import urllib.request
    urllib.request.urlretrieve(url_video_sampel, input_video_path)
    if os.path.exists(input_video_path):
        print(f"✅ Video sampel berhasil disiapkan: {input_video_path} ({os.path.getsize(input_video_path)} bytes)")
except Exception as e:
    print(f"❌ Gagal menyiapkan video sampel: {str(e)}")

print("\n" + "="*60 + "\n")

# 2. Eksekusi Pengujian Modul Inti (Vision + Audio Pipeline)
print("🧠 Langkah 2: Menjalankan Core Logic Pipeline AI...")
print("             (Memproses Audio via FFmpeg & Wajah via MediaPipe BlazeFace)\n")

try:
    # Memanggil fungsi utama dari modul filter_faceless Anda
    # Catatan: Sesuaikan variabel return jika fungsi Anda mengembalikan data berbeda (misal tuple)
    hasil_pemrosesan = deteksi_faceless_video(input_video_path)

    print("\n" + "="*60 + "\n")
    print("🎉 Langkah 3: Hasil Evaluasi Skenario Testing")

    if isinstance(hasil_pemrosesan, tuple):
        video_output, log_moderasi = hasil_pemrosesan
        print("📍 STATUS OUTCOME : SUCCESS (Tuple Data Returned)")
        print(f"📂 File Video Hasil Blur : {video_output}")
        print(f"📄 Eksistensi File Video : {'Ada (Siap Dimainkan)' if os.path.exists(video_output) else 'Tidak Ditemukan'}")
        print("\n📝 ISI LOG KONSOL MODERASI AI:")
        print("-" * 40)
        print(log_moderasi)
        print("-" * 40)
    else:
        print("📍 STATUS OUTCOME : SUCCESS (Single Data Returned)")
        print(f"📂 Jalur Output/Log : {hasil_pemrosesan}")
        if os.path.exists(str(hasil_pemrosesan)):
            print("📄 Eksistensi File      : Ada dan Berhasil Terbentuk.")

except Exception as e:
    print(f"\n❌ Skenario Testing Gagal di Tengah Jalan!")
    print(f"Detail Eror: {str(e)}")
    print("\n💡 Tips Solusi:")
    print("1. Pastikan library sistem 'ffmpeg' sudah terinstal di Colab (jalankan: !apt-get install ffmpeg).")
    print("2. Pastikan parameter input fungsi 'deteksi_faceless_video' di filter_faceless.py sesuai dengan skrip ini.")

# 📋 SKENARIO TESTING 2: AUDIO-ONLY & MUSIC DETECTION (NO HUMAN FACE)

In [ ]:
import os
import urllib.request
from filter_faceless import deteksi_faceless_video

# =====================================================================
# 📋 SKENARIO TESTING 2 (PERBAIKAN): AUDIO/MUSIC ONLY (NO HUMAN FACE)
# =====================================================================

# 1. Gunakan URL sampel video alternatif yang dijamin aktif (OpenCV/FFmpeg Test Media)
url_video_musik_alt = 'https://raw.githubusercontent.com/opencv/opencv/master/samples/data/vtest.avi'
input_video_musik_path = 'video_uji_coba_musik.mp4'

print("⏳ Langkah 1: Mengunduh video sampel musik alternatif (Tanpa Wajah)...")
try:
    # Menambahkan User-Agent agar tidak diblokir oleh server keamanan GitHub
    opener = urllib.request.build_opener()
    opener.addheaders = [('User-agent', 'Mozilla/5.0')]
    urllib.request.install_opener(opener)

    urllib.request.urlretrieve(url_video_musik_alt, input_video_musik_path)
    if os.path.exists(input_video_musik_path):
        print(f"✅ Video sampel musik siap: {input_video_musik_path} ({os.path.getsize(input_video_musik_path)} bytes)")
except Exception as e:
    print(f"❌ Gagal menyiapkan video sampel: {str(e)}")

print("\n" + "="*60 + "\n")

# 2. Eksekusi Pengujian Modul untuk Skenario Non-Wajah + Deteksi Musik
print("🧠 Langkah 2: Menjalankan Core Logic Pipeline AI untuk Skenario 2...\n")

if os.path.exists(input_video_musik_path):
    try:
        hasil_pemrosesan_2 = deteksi_faceless_video(input_video_musik_path)

        print("\n" + "="*60 + "\n")
        print("🎉 Langkah 3: Hasil Evaluasi Skenario Testing 2")

        if isinstance(hasil_pemrosesan_2, tuple):
            video_output_2, log_moderasi_2 = hasil_pemrosesan_2
            print("📍 STATUS OUTCOME : SUCCESS")
            print("\n📝 ISI LOG KONSOL MODERASI AI (SKENARIO AUDIO/MUSIK):")
            print("-" * 40)
            print(log_moderasi_2)
            print("-" * 40)
        else:
            print("📍 STATUS OUTCOME : SUCCESS")
            print(f"📂 Jalur Output/Log : {hasil_pemrosesan_2}")
            print("\n💡 Silakan cek log cetakan di atas untuk memastikan status deteksi musik Anda.")

    except Exception as e:
        print(f"\n❌ Skenario Testing 2 Gagal di Tengah Eksekusi Pipeline!")
        print(f"Detail Eror: {str(e)}")
else:
    print("❌ Menghentikan pengujian karena file video tidak berhasil didownload.")

# 📋 SKENARIO TESTING 3: REJECT VIDEO (MUSIC DETECTED / AUDIO TOO LOUD)

In [ ]:
import os
import urllib.request
from filter_faceless import deteksi_faceless_video

# =====================================================================
# 📋 SKENARIO TESTING 3: REJECT VIDEO (MUSIC DETECTED / AUDIO TOO LOUD)
# =====================================================================

# 1. Gunakan sampel video yang memiliki audio/musik bising tanpa wajah (rekaman audio sintetis/test tone)
url_video_loud = 'https://raw.githubusercontent.com/mdn/learning-area/main/html/multimedia-and-embedding/video-and-audio-content/rabbit320.mp4'
input_video_reject_path = 'video_uji_coba_reject.mp4'

print("⏳ Langkah 1: Mengunduh video sampel musik bising...")
try:
    opener = urllib.request.build_opener()
    opener.addheaders = [('User-agent', 'Mozilla/5.0')]
    urllib.request.install_opener(opener)

    urllib.request.urlretrieve(url_video_loud, input_video_reject_path)
    if os.path.exists(input_video_reject_path):
        print(f"✅ Video sampel bising siap: {input_video_reject_path} ({os.path.getsize(input_video_reject_path)} bytes)")
except Exception as e:
    print(f"❌ Gagal menyiapkan video sampel: {str(e)}")

print("\n" + "="*60 + "\n")

# 2. Eksekusi Pengujian Modul untuk Skenario REJECT
print("🧠 Langkah 2: Menjalankan Core Logic Pipeline AI untuk Skenario REJECT...\n")

if os.path.exists(input_video_reject_path):
    try:
        hasil_pemrosesan_3 = deteksi_faceless_video(input_video_reject_path)

        print("\n" + "="*60 + "\n")
        print("🎉 Langkah 3: Hasil Evaluasi Skenario REJECT")

        if isinstance(hasil_pemrosesan_3, tuple):
            video_output_3, log_moderasi_3 = hasil_pemrosesan_3
            print("📍 STATUS OUTCOME : SUCCESS")
            print("\n📝 ISI LOG KONSOL MODERASI AI (SKENARIO REJECT):")
            print("-" * 40)
            print(log_moderasi_3)
            print("-" * 40)
        else:
            print("📍 STATUS OUTCOME : SUCCESS")
            print(f"📂 Jalur Output/Log : {hasil_pemrosesan_3}")
            print("\n💡 Silakan cek log di atas. Pastikan status akhir agen adalah REJECTED 🔴 karena faktor audio/musik.")

    except Exception as e:
        print(f"\n❌ Skenario Testing 3 Gagal di Tengah Eksekusi!")
        print(f"Detail Eror: {str(e)}")
else:
    print("❌ Menghentikan pengujian karena file video tidak ditemukan.")

# 📋 SKENARIO TESTING 3 (FORCED REJECT): INJECT HIGH DISTORTION AUDIO

In [ ]:
import urllib.request
import os
import sys

# URL untuk model Mediapipe BlazeFace Full Range
model_url = "https://storage.googleapis.com/mediapipe-models/face_detector/blaze_face_full_range/float16/1/blaze_face_full_range.tflite"
model_filename = "blaze_face_full_range.tflite" # Updated filename

print(f"⏳ Mengunduh model AI Face Detection: {model_filename}...")

try:
    urllib.request.urlretrieve(model_url, model_filename)
    if os.path.exists(model_filename):
        print(f"✅ Model '{model_filename}' berhasil diunduh ke {os.getcwd()}.")
    else:
        print(f"❌ Gagal mengunduh model '{model_filename}'.")
except Exception as e:
    print(f"❌ Terjadi kesalahan saat mengunduh model: {e}")

In [ ]:
import importlib
import sys

# Force a complete re-import of filter_faceless to clear any stubborn cache
# and ensure the new model path is picked up.
if 'filter_faceless' in sys.modules:
    del sys.modules['filter_faceless']

# Now, import the module fresh to reflect any changes on disk.
import filter_faceless
from filter_faceless import deteksi_faceless_video

print("✅ Modul 'filter_faceless' berhasil dimuat ulang setelah pengunduhan model!")

---

Sekarang, mari kita coba jalankan kembali Gradio UI dengan modul yang sudah diperbarui.

In [ ]:
import gradio as gr
import os
import sys
import nest_asyncio

nest_asyncio.apply()

# Ensure the module is reloaded to reflect the latest changes
# This is redundant if the previous cell was run, but good for robustness.
if 'filter_faceless' in sys.modules:
    del sys.modules['filter_faceless']
from filter_faceless import deteksi_faceless_video

def gradio_deteksi_faceless_video(video_input_path, audio_input_path=None):
    """Wrapper function for Gradio interface."""
    if not video_input_path:
        return None, "❌ Harap unggah file video!"

    print(f"\nProcessing video: {video_input_path}")
    if audio_input_path:
        print(f"Injecting custom audio from: {audio_input_path}")

    try:
        result = deteksi_faceless_video(video_input_path, inject_audio_path=audio_input_path)

        if isinstance(result, tuple):
            output_video_path, moderation_log = result
        else:
            output_video_path = result # Assume it's the video path if not tuple
            moderation_log = "Log moderasi dicetak langsung ke konsol atau tidak tersedia dalam format return." # Fallback

        log_summary = f"Processing finished!\nOutput Video: {output_video_path}"
        if 'REJECTED' in output_video_path.upper():
            log_summary += "\nStatus: REJECTED"
        elif 'APPROVED' in output_video_path.upper():
            log_summary += "\nStatus: APPROVED"
        else:
            log_summary += "\nStatus: Unknown (Check console for full log)"

        return output_video_path, log_summary + "\n\n(Detail log lengkap juga ada di konsol Colab.)"

    except Exception as e:
        return None, f"❌ Error during video processing: {str(e)}"

# Gradio Interface
with gr.Blocks() as demo:
    gr.Markdown(
        """
        # Multimedia Privacy Agent
        Unggah video Anda dan secara opsional tambahkan audio kustom untuk diproses oleh agen privasi.
        Agen akan mengaburkan wajah dan mendeteksi musik/suara bising.
        """
    )
    with gr.Row():
        video_input = gr.Video(label="Unggah Video (Wajib)")
        audio_input = gr.Audio(type="filepath", label="Unggah Audio Kustom (Opsional)")

    with gr.Row():
        process_button = gr.Button("Proses Video")

    with gr.Row():
        video_output = gr.Video(label="Video Hasil Proses")
        log_output = gr.Textbox(label="Log Moderasi AI")

    process_button.click(
        fn=gradio_deteksi_faceless_video,
        inputs=[video_input, audio_input],
        outputs=[video_output, log_output]
    )

gr.close_all() # Close any existing Gradio apps
demo.launch(debug=True, share=True)

---

Sekarang, mari kita coba jalankan kembali Gradio UI dengan model yang sudah tersedia.

In [ ]:
import gradio as gr
import os
import sys

# Ensure the module is reloaded to reflect the latest changes
if 'filter_faceless' in sys.modules:
    del sys.modules['filter_faceless']
from filter_faceless import deteksi_faceless_video

def gradio_deteksi_faceless_video(video_input_path, audio_input_path=None):
    """Wrapper function for Gradio interface."""
    if not video_input_path:
        return None, "❌ Harap unggah file video!"

    print(f"\nProcessing video: {video_input_path}")
    if audio_input_path:
        print(f"Injecting custom audio from: {audio_input_path}")

    try:
        # The deteksi_faceless_video function prints logs directly
        # and returns the output video path. If it returns a tuple,
        # we need to handle it.
        result = deteksi_faceless_video(video_input_path, inject_audio_path=audio_input_path)

        if isinstance(result, tuple):
            output_video_path, moderation_log = result
        else:
            output_video_path = result # Assume it's the video path if not tuple
            moderation_log = "Log moderasi dicetak langsung ke konsol atau tidak tersedia dalam format return." # Fallback

        # Since the original function prints the log, we need to capture or re-read it if possible.
        # For simplicity here, we'll indicate success and rely on console output for detailed logs.
        # If the function were modified to return the log string, it would be better.

        # Let's create a simplified log summary based on the output filename if available
        log_summary = f"Processing finished!\nOutput Video: {output_video_path}"
        if 'REJECTED' in output_video_path.upper(): # Simple check for rejection in filename (if any)
            log_summary += "\nStatus: REJECTED"
        elif 'APPROVED' in output_video_path.upper():
            log_summary += "\nStatus: APPROVED"
        else:
            log_summary += "\nStatus: Unknown (Check console for full log)"

        # Placeholder for full log if deteksi_faceless_video doesn't return it directly
        # In a real scenario, you'd modify deteksi_faceless_video to return the log string.
        return output_video_path, log_summary + "\n\n(Detail log lengkap juga ada di konsol Colab.)"

    except Exception as e:
        return None, f"❌ Error during video processing: {str(e)}"

# Gradio Interface
with gr.Blocks() as demo:
    gr.Markdown(
        """
        # Multimedia Privacy Agent
        Unggah video Anda dan secara opsional tambahkan audio kustom untuk diproses oleh agen privasi.
        Agen akan mengaburkan wajah dan mendeteksi musik/suara bising.
        """
    )
    with gr.Row():
        video_input = gr.Video(label="Unggah Video (Wajib)")
        audio_input = gr.Audio(type="filepath", label="Unggah Audio Kustom (Opsional)")

    with gr.Row():
        process_button = gr.Button("Proses Video")

    with gr.Row():
        video_output = gr.Video(label="Video Hasil Proses")
        log_output = gr.Textbox(label="Log Moderasi AI")

    process_button.click(
        fn=gradio_deteksi_faceless_video,
        inputs=[video_input, audio_input],
        outputs=[video_output, log_output]
    )

gr.close_all() # Close any existing Gradio apps
demo.launch(debug=True, share=True)

In [ ]:
# Memastikan FFmpeg terinstal untuk operasi audio/video
!apt-get update -qq
!apt-get install -y ffmpeg

In [ ]:
import os
import urllib.request
from filter_faceless import deteksi_faceless_video
import subprocess

# =====================================================================
# 📋 SKENARIO TESTING 3 (FORCED REJECT): INJECT HIGH DISTORTION AUDIO
# =====================================================================

input_original = 'video_uji_coba_reject.mp4'
input_forced_reject = 'video_forced_music_reject.mp4'
audio_noise_file = 'forced_noise_audio.wav'

print("⏳ Langkah 1: Mempersiapkan Video dengan Polusi Suara Tinggi...")

if os.path.exists(input_original):
    # 1. Get video duration
    try:
        cmd_duration = f"ffprobe -v error -show_entries format=duration -of default=noprint_wrappers=1:nokey=1 {input_original}"
        duration_str = subprocess.check_output(cmd_duration, shell=True, text=True).strip()
        duration_seconds = float(duration_str)
        print(f"✅ Durasi video asli: {duration_seconds:.2f} detik")
    except Exception as e:
        print(f"❌ Gagal mendapatkan durasi video: {e}")
        duration_seconds = 10  # Default duration if fails

    # 2. Generate a loud white noise audio file for the video duration
    # 'anoisesrc' generates white noise, 'a=0.9' sets amplitude, 'r=44100' sets sample rate.
    # '-af volume=20dB' further boosts the volume significantly.
    cmd_generate_noise = f'ffmpeg -y -f lavfi -i "anoisesrc=d={duration_seconds}:a=0.9:r=44100" -af "volume=20dB" {audio_noise_file}'
    os.system(cmd_generate_noise)

    if os.path.exists(audio_noise_file):
        print(f"✅ File audio noise siap: {audio_noise_file}")
    else:
        print("❌ Gagal membuat file audio noise.")

    # 3. Mux the original video stream with the new noisy audio stream
    # This replaces the original audio with the generated white noise.
    cmd_inject_noise = f'ffmpeg -y -i {input_original} -i {audio_noise_file} -map 0:v -map 1:a -c:v copy -shortest {input_forced_reject}'
    os.system(cmd_inject_noise)

    if os.path.exists(input_forced_reject):
        print(f"✅ Video dengan polusi suara ekstrem siap: {input_forced_reject}")
    else:
        print("❌ Gagal menyuntikkan audio bising via FFmpeg.")
else:
    print("❌ File dasar tidak ditemukan. Jalankan cell skenario 3 sebelumnya terlebih dahulu.")

print("\n" + "="*60 + "\n")

# 2. Jalankan Ulang Pipeline AI dengan Video yang Sudah Direkayasa Menjadi Bising
print("🧠 Langkah 2: Menjalankan Core Logic Pipeline AI untuk Forced REJECT...\n")

if os.path.exists(input_forced_reject):
    try:
        hasil_pemrosesan_3 = deteksi_faceless_video(input_forced_reject)

        print("\n" + "="*60 + "\n")
        print("🎉 Langkah 3: Hasil Evaluasi Skenario Forced REJECT")

        if isinstance(hasil_pemrosesan_3, tuple):
            video_output_3, log_moderasi_3 = hasil_pemrosesan_3
            print("📍 STATUS OUTCOME : SUCCESS")
            print("\n📝 ISI LOG KONSOL MODERASI AI (SKENARIO FORCED REJECT):")
            print("-" * 40)
            print(log_moderasi_3)
            print("-" * 40)
        else:
            print("📍 STATUS OUTCOME : SUCCESS")
            print(f"📂 Jalur Output/Log : {hasil_pemrosesan_3}")
            print("\n💡 Silakan cek log di atas. Pastikan status akhir agen adalah REJECTED 🔴 karena faktor audio/musik yang sengaja ditingkatkan.")

    except Exception as e:
        print(f"\n❌ Skenario Testing 3 (Forced REJECT) Gagal di Tengah Eksekusi!")
        print(f"Detail Eror: {str(e)}")
else:
    print("❌ Menghentikan pengujian karena file video yang direkayasa tidak ditemukan.")

# Menambahkan Fungsi Inject Audio Custom Pada Modul

### Panduan Modifikasi `filter_faceless.py`

Untuk menambahkan kemampuan injeksi audio kustom, Anda perlu mengedit file `filter_faceless.py` secara langsung. Ikuti langkah-langkah berikut:

1.  **Buka file `filter_faceless.py`:** Anda bisa melakukannya dari panel file di sebelah kiri Google Colab, atau dengan menggunakan perintah seperti `!cat filter_faceless.py` untuk melihat isinya dan kemudian menggunakan editor teks untuk mengeditnya (misalnya, mengunduh, mengedit, lalu mengunggah kembali, atau menggunakan perintah `%%writefile` jika Anda tahu persis apa yang harus diubah).

2.  **Cari fungsi `deteksi_faceless_video`:** Ini adalah fungsi utama yang akan kita ubah.

3.  **Modifikasi tanda tangan fungsi (function signature):** Tambahkan parameter `inject_audio_path` dengan nilai default `None`.
    
    **Sebelumnya (kemungkinan):**
    ```python
def deteksi_faceless_video(input_video_path):
    # ... kode yang sudah ada ...
    ```
    
    **Setelah dimodifikasi:**
    ```python
def deteksi_faceless_video(input_video_path, inject_audio_path=None):
    # ... kode yang sudah ada ...
    ```

4.  **Tambahkan logika injeksi audio:** Sisipkan kode berikut *setelah* `output_ready.mp4` atau file output utama lainnya (jika namanya berbeda) telah dibuat oleh proses filter, dan *sebelum* fungsi `deteksi_faceless_video` mengembalikan nilainya. Lokasi yang tepat mungkin bervariasi sedikit tergantung struktur internal fungsi Anda, tetapi umumnya akan berada di bagian akhir fungsi setelah semua pemrosesan selesai dan file output video sudah ada.

    ```python
    # --- START Blok Logika Injeksi Audio Kustom ---
    output_video_filename = 'output_ready.mp4' # Pastikan ini sesuai dengan nama file output Anda

    if os.path.exists(output_video_filename) and inject_audio_path and os.path.exists(inject_audio_path):
        print(f"🎵 Mengganti audio di '{output_video_filename}' dengan audio kustom dari '{inject_audio_path}'...")
        temp_video_with_custom_audio = 'temp_output_custom_audio.mp4'
        
        # Perintah FFmpeg untuk mengganti stream audio
        # Ini mengambil stream video dari output_video_filename (map 0:v)
        # dan stream audio dari inject_audio_path (map 1:a)
        # dan menyalinnya ke temp_video_with_custom_audio
        cmd_replace_audio = (
            f'ffmpeg -y -i "{output_video_filename}" -i "{inject_audio_path}" '  # Input video dan input audio kustom
            f'-map 0:v -map 1:a '                                        # Peta stream video dari input 0, stream audio dari input 1
            f'-c:v copy -shortest "{temp_video_with_custom_audio}"'    # Salin codec video, potong jika salah satu stream lebih pendek
        )
        os.system(cmd_replace_audio)
        
        if os.path.exists(temp_video_with_custom_audio):
            os.replace(temp_video_with_custom_audio, output_video_filename)
            print(f"✅ Audio berhasil diganti di '{output_video_filename}' dengan audio kustom.")
        else:
            print("❌ Gagal mengganti audio kustom. Pastikan FFmpeg terinstal dan path file audio benar.")
    # --- END Blok Logika Injeksi Audio Kustom ---
    ```

5.  **Simpan perubahan** pada `filter_faceless.py`.

Setelah Anda selesai mengedit dan menyimpan file, jalankan kembali sel `CjL-NOPR5W8Q` di notebook ini untuk memuat ulang modul yang telah dimodifikasi.

Setelah Anda memodifikasi `filter_faceless.py` dan memuat ulang modulnya, kita bisa menguji fungsionalitas injeksi audio. Pertama, mari kita unduh file audio sampel untuk digunakan.

In [ ]:
import urllib.request
import os

# URL untuk mengunduh sampel audio
# Menggunakan URL alternatif yang lebih stabil untuk MP3
url_sample_audio = 'https://www.soundhelix.com/examples/mp3/SoundHelix-Song-1.mp3' # Contoh MP3
custom_audio_file_path = 'custom_background_audio.mp3' # Mengubah ekstensi file agar sesuai dengan MP3

print(f"⏳ Mengunduh file audio sampel: {custom_audio_file_path}...")
try:
    # Menambahkan User-Agent agar tidak diblokir oleh server keamanan
    opener = urllib.request.build_opener()
    opener.addheaders = [('User-agent', 'Mozilla/5.0')]
    urllib.request.install_opener(opener)

    urllib.request.urlretrieve(url_sample_audio, custom_audio_file_path)
    if os.path.exists(custom_audio_file_path):
        print(f"✅ File audio sampel berhasil diunduh: {custom_audio_file_path}")
    else:
        print("❌ Gagal mengunduh file audio sampel.")
except Exception as e:
    print(f"❌ Terjadi kesalahan saat mengunduh audio: {e}")

Sekarang, dengan modul `filter_faceless` yang sudah dimodifikasi (setelah Anda mengedit file dan me-re-import), dan file audio sampel sudah diunduh, kita bisa menjalankan skenario testing awal (menggunakan `video_uji_coba.mp4`) dan menyuntikkan audio kustom ini.

In [ ]:
import os
import sys

# Force a complete re-import of filter_faceless to clear any stubborn cache
if 'filter_faceless' in sys.modules:
    del sys.modules['filter_faceless']
from filter_faceless import deteksi_faceless_video

# Pastikan file video dan audio kustom ada
input_video_path = 'video_uji_coba.mp4' # Ini adalah video yang digunakan di SKENARIO TESTING 1
custom_audio_file_path = 'custom_background_audio.mp3' # Mengubah ekstensi agar sesuai

if os.path.exists(input_video_path) and os.path.exists(custom_audio_file_path):
    print(f"🧠 Menjalankan `deteksi_faceless_video` dengan injeksi audio kustom...")
    try:
        # Panggil fungsi dengan parameter inject_audio_path
        hasil_pemrosesan_injeksi = deteksi_faceless_video(input_video_path, inject_audio_path=custom_audio_file_path)

        print("\n" + "="*60 + "\n")
        print("🎉 Hasil Evaluasi Skenario Testing dengan Injeksi Audio Kustom")

        if isinstance(hasil_pemrosesan_injeksi, tuple):
            video_output_injeksi, log_moderasi_injeksi = hasil_pemrosesan_injeksi
            print("📍 STATUS OUTCOME : SUCCESS")
            print(f"📂 File Video Hasil (dengan audio kustom): {video_output_injeksi}")
            print("\n📝 ISI LOG KONSOL MODERASI AI (dengan Injeksi Audio Kustom):")
            print("-" * 40)
            print(log_moderasi_injeksi)
            print("-" * 40)
        else:
            print("📍 STATUS OUTCOME : SUCCESS")
            print(f"📂 Jalur Output/Log : {hasil_pemrosesan_injeksi}")

        print("\n💡 Silakan putar file output_ready.mp4 untuk memverifikasi apakah audio kustom Anda telah berhasil disuntikkan.")

    except Exception as e:
        print(f"\n❌ Skenario Testing dengan Injeksi Audio Kustom Gagal!")
        print(f"Detail Eror: {str(e)}")
else:
    print("❌ Menghentikan pengujian karena file video atau audio kustom tidak ditemukan.")
    if not os.path.exists(input_video_path):
        print(f"- {input_video_path} tidak ditemukan.")
    if not os.path.exists(custom_audio_file_path):
        print(f"- {custom_audio_file_path} tidak ditemukan.")

In [ ]:
import os
from filter_faceless import deteksi_faceless_video

input_video_saya = 'video_saya.mp4'

# Pastikan video_saya.mp4 ada di direktori kerja
# Jika belum ada, Anda perlu mengunggahnya atau menyediakan jalur yang benar.

if os.path.exists(input_video_saya):
    print(f"🧠 Menjalankan `deteksi_faceless_video` pada {input_video_saya}...")
    try:
        hasil_pemrosesan_saya = deteksi_faceless_video(input_video_saya)

        print("\n" + "="*60 + "\n")
        print(f"🎉 Hasil Evaluasi untuk {input_video_saya}")

        if isinstance(hasil_pemrosesan_saya, tuple):
            video_output_saya, log_moderasi_saya = hasil_pemrosesan_saya
            print("📍 STATUS OUTCOME : SUCCESS")
            print(f"📂 File Video Hasil: {video_output_saya}")
            print("\n📝 ISI LOG KONSOL MODERASI AI:")
            print("-" * 40)
            print(log_moderasi_saya)
            print("-" * 40)
        else:
            print("📍 STATUS OUTCOME : SUCCESS")
            print(f"📂 Jalur Output/Log : {hasil_pemrosesan_saya}")

    except Exception as e:
        print(f"\n❌ Pengujian untuk {input_video_saya} Gagal!")
        print(f"Detail Eror: {str(e)}")
else:
    print(f"❌ File '{input_video_saya}' tidak ditemukan. Harap pastikan file tersebut ada di direktori yang sama atau berikan path yang benar.")


In [ ]:
import os
import shutil

source_path = '/content/video_saya.mp4'
destination_path = './video_saya.mp4' # Copy to current working directory

if os.path.exists(source_path):
    shutil.copy(source_path, destination_path)
    print(f"✅ File '{source_path}' berhasil disalin ke '{destination_path}'.")
else:
    print(f"❌ Sumber file '{source_path}' tidak ditemukan untuk disalin.")

In [ ]:
with open('filter_faceless.py', 'r') as f:
    print(f.read())

In [ ]:
import sys
# Install Gradio if not already installed
!pip install -q gradio

# UI Pengujian Untuk Test Fungsi Inject Audio Kustom

In [ ]:
import gradio as gr
import os
import sys

# Ensure the module is reloaded to reflect the latest changes
if 'filter_faceless' in sys.modules:
    del sys.modules['filter_faceless']
from filter_faceless import deteksi_faceless_video

def gradio_deteksi_faceless_video(video_input_path, audio_input_path=None):
    """Wrapper function for Gradio interface."""
    if not video_input_path:
        return None, "❌ Harap unggah file video!"

    print(f"\nProcessing video: {video_input_path}")
    if audio_input_path:
        print(f"Injecting custom audio from: {audio_input_path}")

    try:
        # The deteksi_faceless_video function prints logs directly
        # and returns the output video path. If it returns a tuple,
        # we need to handle it.
        result = deteksi_faceless_video(video_input_path, inject_audio_path=audio_input_path)

        if isinstance(result, tuple):
            output_video_path, moderation_log = result
        else:
            output_video_path = result # Assume it's the video path if not tuple
            moderation_log = "Log moderasi dicetak langsung ke konsol atau tidak tersedia dalam format return." # Fallback

        # Since the original function prints the log, we need to capture or re-read it if possible.
        # For simplicity here, we'll indicate success and rely on console output for detailed logs.
        # If the function were modified to return the log string, it would be better.

        # Let's create a simplified log summary based on the output filename if available
        log_summary = f"Processing finished!\nOutput Video: {output_video_path}"
        if 'REJECTED' in output_video_path.upper(): # Simple check for rejection in filename (if any)
            log_summary += "\nStatus: REJECTED"
        elif 'APPROVED' in output_video_path.upper():
            log_summary += "\nStatus: APPROVED"
        else:
            log_summary += "\nStatus: Unknown (Check console for full log)"

        # Placeholder for full log if deteksi_faceless_video doesn't return it directly
        # In a real scenario, you'd modify deteksi_faceless_video to return the log string.
        return output_video_path, log_summary + "\n\n(Detail log lengkap juga ada di konsol Colab.)"

    except Exception as e:
        return None, f"❌ Error during video processing: {str(e)}"

# Gradio Interface
with gr.Blocks() as demo:
    gr.Markdown(
        """
        # Multimedia Privacy Agent
        Unggah video Anda dan secara opsional tambahkan audio kustom untuk diproses oleh agen privasi.
        Agen akan mengaburkan wajah dan mendeteksi musik/suara bising.
        """
    )
    with gr.Row():
        video_input = gr.Video(label="Unggah Video (Wajib)")
        audio_input = gr.Audio(type="filepath", label="Unggah Audio Kustom (Opsional)")

    with gr.Row():
        process_button = gr.Button("Proses Video")

    with gr.Row():
        video_output = gr.Video(label="Video Hasil Proses")
        log_output = gr.Textbox(label="Log Moderasi AI")

    process_button.click(
        fn=gradio_deteksi_faceless_video,
        inputs=[video_input, audio_input],
        outputs=[video_output, log_output]
    )

gr.close_all() # Close any existing Gradio apps
demo.launch(debug=True, share=True)

In [ ]:
import gradio as gr
import os
import sys

# Ensure the module is reloaded to reflect the latest changes
if 'filter_faceless' in sys.modules:
    del sys.modules['filter_faceless']
from filter_faceless import deteksi_faceless_video

def gradio_deteksi_faceless_video(video_input_path, audio_input_path=None):
    """Wrapper function for Gradio interface."""
    if not video_input_path:
        return None, "❌ Harap unggah file video!"

    print(f"\nProcessing video: {video_input_path}")
    if audio_input_path:
        print(f"Injecting custom audio from: {audio_input_path}")

    try:
        # The deteksi_faceless_video function prints logs directly
        # and returns the output video path. If it returns a tuple,
        # we need to handle it.
        result = deteksi_faceless_video(video_input_path, inject_audio_path=audio_input_path)

        if isinstance(result, tuple):
            output_video_path, moderation_log = result
        else:
            output_video_path = result # Assume it's the video path if not tuple
            moderation_log = "Log moderasi dicetak langsung ke konsol atau tidak tersedia dalam format return." # Fallback

        # Since the original function prints the log, we need to capture or re-read it if possible.
        # For simplicity here, we'll indicate success and rely on console output for detailed logs.
        # If the function were modified to return the log string, it would be better.

        # Let's create a simplified log summary based on the output filename if available
        log_summary = f"Processing finished!\nOutput Video: {output_video_path}"
        if 'REJECTED' in output_video_path.upper(): # Simple check for rejection in filename (if any)
            log_summary += "\nStatus: REJECTED"
        elif 'APPROVED' in output_video_path.upper():
            log_summary += "\nStatus: APPROVED"
        else:
            log_summary += "\nStatus: Unknown (Check console for full log)"

        # Placeholder for full log if deteksi_faceless_video doesn't return it directly
        # In a real scenario, you'd modify deteksi_faceless_video to return the log string.
        return output_video_path, log_summary + "\n\n(Detail log lengkap juga ada di konsol Colab.)"

    except Exception as e:
        return None, f"❌ Error during video processing: {str(e)}"

# Gradio Interface
with gr.Blocks() as demo:
    gr.Markdown(
        """
        # Multimedia Privacy Agent
        Unggah video Anda dan secara opsional tambahkan audio kustom untuk diproses oleh agen privasi.
        Agen akan mengaburkan wajah dan mendeteksi musik/suara bising.
        """
    )
    with gr.Row():
        video_input = gr.Video(label="Unggah Video (Wajib)")
        audio_input = gr.Audio(type="filepath", label="Unggah Audio Kustom (Opsional)")

    with gr.Row():
        process_button = gr.Button("Proses Video")

    with gr.Row():
        video_output = gr.Video(label="Video Hasil Proses")
        log_output = gr.Textbox(label="Log Moderasi AI")

    process_button.click(
        fn=gradio_deteksi_faceless_video,
        inputs=[video_input, audio_input],
        outputs=[video_output, log_output]
    )

gr.close_all() # Close any existing Gradio apps
demo.launch(debug=True, share=True)

In [ ]:
import gradio as gr
import os

# Ensure the module is reloaded to reflect the latest changes
if 'filter_faceless' in sys.modules:
    del sys.modules['filter_faceless']
from filter_faceless import deteksi_faceless_video

def gradio_deteksi_faceless_video(video_input_path, audio_input_path=None):
    """Wrapper function for Gradio interface."""
    if not video_input_path:
        return None, "❌ Harap unggah file video!"

    print(f"\nProcessing video: {video_input_path}")
    if audio_input_path:
        print(f"Injecting custom audio from: {audio_input_path}")

    try:
        # The deteksi_faceless_video function prints logs directly
        # and returns the output video path. If it returns a tuple,
        # we need to handle it.
        result = deteksi_faceless_video(video_input_path, inject_audio_path=audio_input_path)

        if isinstance(result, tuple):
            output_video_path, moderation_log = result
        else:
            output_video_path = result # Assume it's the video path if not tuple
            moderation_log = "Log moderasi dicetak langsung ke konsol atau tidak tersedia dalam format return." # Fallback

        # Since the original function prints the log, we need to capture or re-read it if possible.
        # For simplicity here, we'll indicate success and rely on console output for detailed logs.
        # If the function were modified to return the log string, it would be better.

        # Let's create a simplified log summary based on the output filename if available
        log_summary = f"Processing finished!\nOutput Video: {output_video_path}"
        if 'REJECTED' in output_video_path.upper(): # Simple check for rejection in filename (if any)
            log_summary += "\nStatus: REJECTED"
        elif 'APPROVED' in output_video_path.upper():
            log_summary += "\nStatus: APPROVED"
        else:
            log_summary += "\nStatus: Unknown (Check console for full log)"

        # Placeholder for full log if deteksi_faceless_video doesn't return it directly
        # In a real scenario, you'd modify deteksi_faceless_video to return the log string.
        return output_video_path, log_summary + "\n\n(Detail log lengkap juga ada di konsol Colab.)"

    except Exception as e:
        return None, f"❌ Error during video processing: {str(e)}"

# Gradio Interface
with gr.Blocks() as demo:
    gr.Markdown(
        """
        # Multimedia Privacy Agent
        Unggah video Anda dan secara opsional tambahkan audio kustom untuk diproses oleh agen privasi.
        Agen akan mengaburkan wajah dan mendeteksi musik/suara bising.
        """
    )
    with gr.Row():
        video_input = gr.Video(label="Unggah Video (Wajib)")
        audio_input = gr.Audio(type="filepath", label="Unggah Audio Kustom (Opsional)")

    with gr.Row():
        process_button = gr.Button("Proses Video")

    with gr.Row():
        video_output = gr.Video(label="Video Hasil Proses")
        log_output = gr.Textbox(label="Log Moderasi AI")

    process_button.click(
        fn=gradio_deteksi_faceless_video,
        inputs=[video_input, audio_input],
        outputs=[video_output, log_output]
    )

demo.launch(debug=True, share=True)

In [ ]:
from filter_faceless import deteksi_faceless_video

input_video_saya = 'video_saya.mp4'

# Memastikan file video_saya.mp4 ada di direktori kerja
if os.path.exists(input_video_saya):
    print(f"🧠 Menjalankan `deteksi_faceless_video` pada {input_video_saya}...")
    try:
        hasil_pemrosesan_saya = deteksi_faceless_video(input_video_saya)

        print("\n" + "="*60 + "\n")
        print(f"🎉 Hasil Evaluasi untuk {input_video_saya}")

        if isinstance(hasil_pemrosesan_saya, tuple):
            video_output_saya, log_moderasi_saya = hasil_pemrosesan_saya
            print("📍 STATUS OUTCOME : SUCCESS")
            print(f"📂 File Video Hasil: {video_output_saya}")
            print("\n📝 ISI LOG KONSOL MODERASI AI:")
            print("-" * 40)
            print(log_moderasi_saya)
            print("-" * 40)
        else:
            print("📍 STATUS OUTCOME : SUCCESS")
            print(f"📂 Jalur Output/Log : {hasil_pemrosesan_saya}")

    except Exception as e:
        print(f"\n❌ Pengujian untuk {input_video_saya} Gagal!")
        print(f"Detail Eror: {str(e)}")
else:
    print(f"❌ File '{input_video_saya}' tidak ditemukan. Harap pastikan file tersebut ada di direktori yang sama atau berikan path yang benar.")

In [ ]:
import cv2
import matplotlib.pyplot as plt
import os

output_video_path = 'output_ready.mp4'

if os.path.exists(output_video_path):
    print(f"✅ Membaca video hasil proses: {output_video_path}")
    cap = cv2.VideoCapture(output_video_path)

    if not cap.isOpened():
        print(f"❌ Gagal membuka video: {output_video_path}")
    else:
        # Baca dan tampilkan beberapa frame untuk verifikasi
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        frames_to_show = [0, int(cap.get(cv2.CAP_PROP_FRAME_COUNT) / 2), int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) - 1]

        for i, frame_idx in enumerate(frames_to_show):
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
            ret, frame = cap.read()
            if ret:
                # OpenCV membaca dalam BGR, matplotlib menampilkan dalam RGB
                frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                axes[i].imshow(frame_rgb)
                axes[i].set_title(f'Frame {frame_idx}')
                axes[i].axis('off')
            else:
                axes[i].set_title(f'Frame {frame_idx} (Gagal Dibaca)')
                axes[i].axis('off')
        plt.tight_layout()
        plt.show()

        cap.release()
        print("💡 Video berhasil dibaca. Anda bisa mengunduh 'output_ready.mp4' untuk melihat video lengkapnya.")
else:
    print(f"❌ File video '{output_video_path}' tidak ditemukan. Pastikan sudah ada yang diproses.")

In [ ]:
import os

file_to_check = 'video_uji_coba_reject.mp4'
if os.path.exists(file_to_check):
    print(f"✅ File '{file_to_check}' ditemukan di {os.getcwd()}.")
else:
    print(f"❌ File '{file_to_check}' TIDAK ditemukan di {os.getcwd()}.")

In [ ]:
%cd /content/ai-multimedia-privacy-agent/

In [ ]:
import importlib
import sys

# Force a complete re-import of filter_faceless to clear any stubborn cache
# and ensure the new model path is picked up.
if 'filter_faceless' in sys.modules:
    del sys.modules['filter_faceless']

# Now, import the module fresh to reflect any changes on disk.
import filter_faceless
from filter_faceless import deteksi_faceless_video

print("✅ Modul 'filter_faceless' berhasil dimuat ulang!")

---

Sekarang, mari kita jalankan kembali Gradio UI dengan modul yang sudah diperbarui.

In [ ]:
import gradio as gr
import os
import sys
import nest_asyncio

nest_asyncio.apply()

# Ensure the module is reloaded to reflect the latest changes
# This is redundant if the previous cell was run, but good for robustness.
if 'filter_faceless' in sys.modules:
    del sys.modules['filter_faceless']
from filter_faceless import deteksi_faceless_video

def gradio_deteksi_faceless_video(video_input_path, audio_input_path=None):
    """Wrapper function for Gradio interface."""
    if not video_input_path:
        return None, "❌ Harap unggah file video!"

    print(f"\nProcessing video: {video_input_path}")
    if audio_input_path:
        print(f"Injecting custom audio from: {audio_input_path}")

    try:
        result = deteksi_faceless_video(video_input_path, inject_audio_path=audio_input_path)

        if isinstance(result, tuple):
            output_video_path, moderation_log = result
        else:
            output_video_path = result # Assume it's the video path if not tuple
            moderation_log = "Log moderasi dicetak langsung ke konsol atau tidak tersedia dalam format return." # Fallback

        log_summary = f"Processing finished!\nOutput Video: {output_video_path}"
        if 'REJECTED' in output_video_path.upper():
            log_summary += "\nStatus: REJECTED"
        elif 'APPROVED' in output_video_path.upper():
            log_summary += "\nStatus: APPROVED"
        else:
            log_summary += "\nStatus: Unknown (Check console for full log)"

        return output_video_path, log_summary + "\n\n(Detail log lengkap juga ada di konsol Colab.)"

    except Exception as e:
        return None, f"❌ Error during video processing: {str(e)}"

# Gradio Interface
with gr.Blocks() as demo:
    gr.Markdown(
        """
        # Multimedia Privacy Agent
        Unggah video Anda dan secara opsional tambahkan audio kustom untuk diproses oleh agen privasi.
        Agen akan mengaburkan wajah dan mendeteksi musik/suara bising.
        """
    )
    with gr.Row():
        video_input = gr.Video(label="Unggah Video (Wajib)")
        audio_input = gr.Audio(type="filepath", label="Unggah Audio Kustom (Opsional)")

    with gr.Row():
        process_button = gr.Button("Proses Video")

    with gr.Row():
        video_output = gr.Video(label="Video Hasil Proses")
        log_output = gr.Textbox(label="Log Moderasi AI")

    process_button.click(
        fn=gradio_deteksi_faceless_video,
        inputs=[video_input, audio_input],
        outputs=[video_output, log_output]
    )

gr.close_all() # Close any existing Gradio apps
demo.launch(debug=True, share=True)

In [ ]:
import gradio as gr
import os
import sys
import nest_asyncio

nest_asyncio.apply()

# Ensure the module is reloaded to reflect the latest changes
# This is redundant if the previous cell was run, but good for robustness.
if 'filter_faceless' in sys.modules:
    del sys.modules['filter_faceless']
from filter_faceless import deteksi_faceless_video

def gradio_deteksi_faceless_video(video_input_path, audio_input_path=None):
    """Wrapper function for Gradio interface."""
    if not video_input_path:
        return None, "❌ Harap unggah file video!"

    print(f"\nProcessing video: {video_input_path}")
    if audio_input_path:
        print(f"Injecting custom audio from: {audio_input_path}")

    try:
        result = deteksi_faceless_video(video_input_path, inject_audio_path=audio_input_path)

        if isinstance(result, tuple):
            output_video_path, moderation_log = result
        else:
            output_video_path = result # Assume it's the video path if not tuple
            moderation_log = "Log moderasi dicetak langsung ke konsol atau tidak tersedia dalam format return." # Fallback

        log_summary = f"Processing finished!\nOutput Video: {output_video_path}"
        if 'REJECTED' in output_video_path.upper():
            log_summary += "\nStatus: REJECTED"
        elif 'APPROVED' in output_video_path.upper():
            log_summary += "\nStatus: APPROVED"
        else:
            log_summary += "\nStatus: Unknown (Check console for full log)"

        return output_video_path, log_summary + "\n\n(Detail log lengkap juga ada di konsol Colab.)"

    except Exception as e:
        return None, f"❌ Error during video processing: {str(e)}"

# Gradio Interface
with gr.Blocks() as demo:
    gr.Markdown(
        """
        # Multimedia Privacy Agent
        Unggah video Anda dan secara opsional tambahkan audio kustom untuk diproses oleh agen privasi.
        Agen akan mengaburkan wajah dan mendeteksi musik/suara bising.
        """
    )
    with gr.Row():
        video_input = gr.Video(label="Unggah Video (Wajib)")
        audio_input = gr.Audio(type="filepath", label="Unggah Audio Kustom (Opsional)")

    with gr.Row():
        process_button = gr.Button("Proses Video")

    with gr.Row():
        video_output = gr.Video(label="Video Hasil Proses")
        log_output = gr.Textbox(label="Log Moderasi AI")

    process_button.click(
        fn=gradio_deteksi_faceless_video,
        inputs=[video_input, audio_input],
        outputs=[video_output, log_output]
    )

gr.close_all() # Close any existing Gradio apps
demo.launch(debug=True, share=True)

## Membersihkan File Hasil

In [ ]:
import os

# Daftar file yang dihasilkan selama eksekusi yang perlu dihapus
files_to_remove = [
    'output_ready.mp4',
    'output_faceless.mp4',
    'temp_audio.wav',
    'temp_output_custom_audio.mp4',
    'forced_noise_audio.wav',
    'video_forced_music_reject.mp4',
    'video_uji_coba.mp4',
    'video_uji_coba_musik.mp4',
    'video_uji_coba_reject.mp4',
    'custom_background_audio.mp3',
    'blaze_face_full_range.tflite'
]

print("⏳ Membersihkan file-file yang dihasilkan...")
for file_name in files_to_remove:
    if os.path.exists(file_name):
        os.remove(file_name)
        print(f"✅ Berhasil menghapus: {file_name}")
    else:
        print(f"ℹ️ File tidak ditemukan (sudah bersih atau tidak pernah dibuat): {file_name}")

print("✅ Pembersihan selesai.")

## Menyimpan Hasil Kerja ke GitHub

Untuk menyimpan notebook ini ke repositori GitHub Anda, ikuti langkah-langkah berikut:

1.  **Unduh Notebook:**
    *   Pergi ke `File` (Berkas) di menu atas Google Colab.
    *   Pilih `Save a copy in GitHub` (Simpan salinan di GitHub). Anda akan diminta untuk mengautentikasi GitHub Anda dan memilih repositori serta nama file.

    *Atau Anda bisa mengunduhnya terlebih dahulu, lalu mengunggahnya secara manual:*
    *   Pilih `Download` (Unduh) > `Download .ipynb` (Unduh .ipynb).

2.  **Unggah ke Repositori GitHub Anda (Jika Diunduh Manual):**
    *   Buka repositori GitHub Anda di browser web.
    *   Arahkan ke folder tempat Anda ingin menyimpan notebook.
    *   Klik tombol `Add file` (Tambahkan file) > `Upload files` (Unggah file).
    *   Seret dan jatuhkan file `.ipynb` yang telah Anda unduh ke area yang ditentukan, atau klik `choose your files` (pilih file Anda) dan pilih file tersebut.
    *   Tambahkan pesan *commit* (misalnya, "Menambahkan notebook final"), lalu klik `Commit changes` (Commit perubahan).

Dengan cara ini, semua perubahan dan hasil kerja Anda akan tersimpan di GitHub.